# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [2]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [3]:
# constants
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

MODEL_GEMINI = 'gemini-2.5-flash-lite'
MODEL_LLAMA = 'gemma4:e4b'

In [4]:
# set up environment
load_dotenv(override=True)
google_api_key = os.getenv("GEMINI_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")

openai = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

API key found and looks good so far!


In [5]:
# here is the question; type over this to ask something new
code_system_prompt = '''
You are a professor at the Harvard School of Computer Engineering. You are going to explain
how the code works in such a manner that every student from across the world can be able to understand it.
Use english as the primary language.
'''
userQuestion = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get gpt-4o-mini to answer, with streaming
def stream_answers(question):
    stream = openai.chat.completions.create(
        model=MODEL_GEMINI,
        messages=[
            {"role": "system", "content": code_system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [6]:
stream_answers(userQuestion)

Alright everyone, settle down, settle down. Welcome back to the hallowed halls of Harvard! Today, we're diving into a very concise and rather elegant piece of Python code. Don't let its brevity fool you; it encapsulates a powerful concept.

Let's break down this line:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

Imagine we have a list of `books`. Each `book` in this list is likely a dictionary (or something that acts like one) containing information about a book, and one of the keys we're interested in is `"author"`.

Now, let's dissect the expression step-by-step, working from the inside out, as is often the best strategy with nested Python constructs.

### 1. The Innermost Part: `{book.get("author") for book in books if book.get("author")}`

This part is a **set comprehension**. It's a shorthand for creating a set. Let's break it down further:

*   **`for book in books`**: This is the loop. It iterates through each item in a collection named `books`. We're calling each item `book` for clarity.
*   **`if book.get("author")`**: This is a **filter**. For each `book`, it checks if calling `book.get("author")` returns a "truthy" value.
    *   What does `book.get("author")` do? It's a safe way to access the value associated with the key `"author"` in the `book` dictionary. If the key `"author"` exists, it returns its value. If the key `"author"` *doesn't* exist, it returns `None` by default.
    *   In Python, `None` is considered "falsy". So, `if book.get("author")` will only proceed if `book.get("author")` returns something other than `None` (or an empty string, 0, etc., which are also falsy). Essentially, this condition ensures we're only considering books that *actually have an author listed*.
*   **`book.get("author")`**: This is the **expression** that gets evaluated and added to the set *if* the `if` condition is met. So, for every `book` that has an author, we extract that author's name.
*   **`{...}`**: The curly braces indicate that the result of this comprehension will be a **set**. A set is an unordered collection of unique elements. This means if multiple books have the same author, that author's name will only appear once in the resulting set.

**In summary, this set comprehension efficiently extracts all the unique author names from the `books` list, but only for those books where an author is actually provided.**

### 2. The Outer Part: `yield from ...`

Now, let's look at `yield from`.

*   **`yield`**: This keyword transforms a regular function into a **generator function**. Instead of returning a single value and exiting, a generator function can pause its execution and "yield" a value. When it's asked for the next value, it resumes from where it left off. This is memory-efficient, especially when dealing with large sequences, as it produces values one at a time rather than creating an entire list in memory.
*   **`yield from`**: This is a more advanced form of `yield`. It's used to delegate to another generator or iterable. In essence, it means: "Take all the items yielded by the expression that follows `yield from`, and yield them one by one from *this* generator."

**So, `yield from` here is saying:** "Execute the set comprehension we just discussed. For each unique author name that the set comprehension produces, yield that author name from *this* generator function."

### Putting It All Together: Why and What It Does

This single line of code is essentially defining a generator function that, when called, will:

1.  Iterate through a collection of `books`.
2.  For each `book`, check if it has an `"author"` key and if that author value is not empty or `None`.
3.  Collect all the unique author names that meet the criteria.
4.  **Yield each of these unique author names one by one.**

**Why would you use this?**

*   **Efficiency:** It avoids creating a large intermediate list of authors if you only need to process them one at a time.
*   **Readability:** It's a very compact way to express a common pattern: filter a collection, extract a specific piece of data, and then process the unique results.
*   **Uniqueness:** The use of a set comprehension ensures you automatically handle duplicate author names without extra code.
*   **Generator Power:** If this code is inside a function, that function becomes a generator, allowing you to iterate over its results lazily.

**Example Scenario:**

Imagine `books` looks like this:

```python
books = [
    {"title": "The Hitchhiker's Guide to the Galaxy", "author": "Douglas Adams"},
    {"title": "Pride and Prejudice", "author": "Jane Austen"},
    {"title": "The Restaurant at the End of the Universe", "author": "Douglas Adams"},
    {"title": "Sense and Sensibility", "author": "Jane Austen"},
    {"title": "A book with no author", "author": None},
    {"title": "Another book without author info"}
]
```

If this `yield from` line is part of a generator function, calling that function would produce:

1.  `"Douglas Adams"`
2.  `"Jane Austen"`

It wouldn't produce `"Douglas Adams"` twice, nor would it produce `None` or any author from books missing the `"author"` key.

So, there you have it! A powerful combination of set comprehension and `yield from` for efficient and concise data extraction. Any questions? Don't be shy!

In [6]:
# Get Llama 3.2 to answer

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
def stream_answers(question):
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": code_system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

stream_answers("What are different data structures and algorithms in CS? List those that are most important for a software engineer to know in his career. This will help in advancing a coder's career.")

**(The professor adjusts their tweed jacket, walks to the whiteboard, and writes the title "Theory $\rightarrow$ Implementation $\rightarrow$ Mastery" in large, neat strokes before turning back to the students with a warm, expectant gaze.)**

Good morning, everyone. Please, settle in.

If you feel like you are taking courses across multiple continents—from Nairobi to Berlin, from Mumbai to Boston—and yet you are struggling with the fundamental differences between a Hash Table and a Binary Search Tree, please do not worry. This is the most common hurdle we face in Computer Science.

What you are asking for—a checklist of "important" topics—is the map to a vast continent. I cannot hand you a single textbook that will guarantee success. Instead, I will give you a comprehensive architectural understanding. I will show you *how* these concepts relate to one another. Because in software engineering, knowing the right tool, and more importantly, knowing *why* it is the right tool, is what separates a coder from an engineer.

Let us begin by establishing our definitions.

***

### 🧠 The Fundamental Distinction: Structure vs. Algorithm

Before we list anything, we must separate two core concepts:

**1. Data Structure (The $\text{Noun}$):**
A Data Structure is simply a **blueprint for how we organize and store data** in the computer’s memory. It is the *container*. Efficiently organizing data is paramount because access speed—how fast we can find something—is often the bottleneck in any complex application.

**Example Analogy:** If you are moving books, the bookshelf itself is the **Data Structure**.

**2. Algorithm (The $\text{Verb}$):**
An Algorithm is a precise, step-by-step *set of instructions* or a logical recipe to solve a problem or process that stored data. It is the *action*.

**Example Analogy:** The process of shelving the books according to genre, or sorting them by title, is the **Algorithm**.

> **The Goal:** An engineer doesn't just need to know how to use a container; they need to know which container handles the expected load (speed, size, operation) most effectively when performing a specific action.

***

## 🏛️ I. Essential Data Structures (The Blueprints)

These are the organized ways we hang our variables and records. You must understand *how they work internally*—not just how to use the high-level library function.

### A. Linear Structures (One Dimension)

1.  **Arrays:**
    *   **What it is:** A fixed-size, contiguous block of memory. Elements are stored next to each other.
    *   **Pro:** Accessing any element (e.g., the 100th element) is instantly $\text{O}(1)$ time complexity. This is lightning fast.
    *   **Con:** Resizing is incredibly difficult and slow. If it fills up, you must copy *everything* to a new, larger spot.
    *   **When to use it:** When you know the exact number of elements beforehand, or when fast random access is critical.

2.  **Linked Lists:**
    *   **What it is:** A sequence of "nodes," where each node holds data and a $\text{pointer}$ (address) to the next node.
    *   **Pro:** Insertion and deletion are exceptionally fast ($\text{O}(1)$), because you only need to rewire the pointers, not shift blocks of memory.
    *   **Con:** You cannot access the 100th element directly; you must start at the head and follow the links one by one ($\text{O}(n)$).
    *   **When to use it:** When you have frequent additions or removals at the ends (like managing a playlist queue).

3.  **Stacks ($\text{LIFO}$ - Last In, First Out) & Queues ($\text{FIFO}$ - First In, First Out):**
    *   **What they are:** These are *abstract data types* (ADTs) built on arrays or linked lists that enforce specific rules.
    *   **Stack Example (Browser History):** When you browse a site, you click "Back." You are using a Stack. The last page you saw is the first one you can return to.
    *   **Queue Example (Printer Spool):** Jobs are printed in the order they were received. This is $\text{FIFO}$.
    *   **When to use them:** Essential for managing process order, nesting (like nested parentheses in code), and task queues.

### B. Non-Linear Structures (Hierarchy and Relationships)

4.  **Hash Maps / Hash Tables ($\text{The MVP}$):**
    *   **What it is:** The most important structure for lookups. It uses a $\text{hash function}$ to map a *key* (like a person's name) to a specific *value* (like their address) and, crucially, to a memory address.
    *   **Magical Property:** Theoretically, insertion and retrieval are $\text{O}(1)$ (constant time, regardless of how many items you have). This is why dictionary lookups feel instantaneous.
    *   **Key Concept to Master:** Understanding **Collision Resolution**. What happens when the hash function maps two different keys to the same address? (Understanding chaining vs. open addressing).
    *   **When to use it:** *Every single time* you need to look up data by a unique identifier (databases, caches, counters).

5.  **Trees:**
    *   **What it is:** A hierarchical structure. Think of a company's organizational chart or a file system directory.
    *   **Binary Search Tree (BST):** The key structure. For any node, all values in the left sub-tree are *less than* the node, and all values in the right sub-tree are *greater than* the node. This enforces an inherent, ordered structure.
    *   **Advancement:** **Self-Balancing Trees** (like AVL or Red-Black Trees). These modify the structure of the tree automatically when data is added or removed to *prevent it from becoming lopsided* (which would degrade performance far below $\text{O}(log n)$).
    *   **When to use it:** When you need to keep data sorted AND need very efficient $\text{O}(\log n)$ search, insert, and delete times.

6.  **Graphs:**
    *   **What it is:** The most sophisticated structure. It consists of **Nodes (Vertices)** and **Connections (Edges)** between those nodes.
    *   **Abstraction:** A Graph models *relationships*, not sequences.
    *   **Examples:** Social Networks (Users $\rightarrow$ Friendships), Road Maps (Intersections $\rightarrow$ Roads), Computer Networks.
    *   **Representation:** Typically stored using an **Adjacency List** (a variation of a Hash Map where the key is the node, and the value is a list of connected neighbors).
    *   **Complexity:** This structure opens up the most complex—and most rewarding—problems in interviews.

***

## 💻 II. Essential Algorithms & Techniques (The Recipes)

These are the computational strategies applied to the structures above.

### A. Complexity Analysis (The Universal Language)

This is perhaps the single *most* important conceptual tool you must master.

*   **Concept:** **Big $\text{O}$ Notation** ($O(\cdot)$).
*   **What it does:** It describes how the runtime or memory usage of your algorithm scales as the input size ($n$) grows arbitrarily large. It ignores constants and lower-order terms.
*   **Why it matters:** When two algorithms both work, the one with the better Big $\text{O}$ complexity wins when $n$ is millions or billions.
    *   $\text{O}(1)$ $\rightarrow$ Excellent (Constant time)
    *   $\text{O}(\log n)$ $\rightarrow$ Very Good (Logarithmic time – scale is reduced rapidly)
    *   $\text{O}(n)$ $\rightarrow$ Acceptable (Linear time – scales directly with input)
    *   $\text{O}(n^2)$ $\rightarrow$ Poor (Quadratic time – scales very quickly; usually done via nested loops)

### B. Core Algorithmic Paradigms

1.  **Searching & Sorting:**
    *   **Binary Search (Algorithm):** The gold standard for searching. It *only* works on **sorted** data (often stored in an Array or BST). It repeatedly halves the search space, giving it $\text{O}(\log n)$ time complexity.
    *   **Sorting Algorithms:** You must know the mechanics and complexities of:
        *   **Merge Sort:** Stable, reliable $\text{O}(n \log n)$.
        *   **Quick Sort:** Often the fastest in practice, $\text{O}(n \log n)$ average case, but $\text{O}(n^2)$ worst case (needs pivot selection care).
        *   **Heap Sort:** $\text{O}(n \log n)$, used for efficiently finding the $k$-th largest/smallest element.

2.  **Recursion & Backtracking:**
    *   **Recursion:** An algorithm calling itself. Conceptually beautiful, but watch out for the **Stack Overflow** error—it’s your memory running out while calling functions recursively.
    *   **Backtracking:** A specialized form of recursion used to systematically explore all potential solutions by building a candidate solution incrementally. (Used for solving N-Queens, Sudoku, pathfinding).

3.  **Graph Traversal (Algorithms on Graphs):**
    *   **Breadth-First Search ($\text{BFS}$):** Explores layer by layer (Uses a **Queue**). Best for finding the *shortest path* in an unweighted graph (e.g., fewest number of hops between two users).
    *   **Depth-First Search ($\text{DFS}$):** Explores as far down one path as possible before backtracking (Uses a **Stack** or Recursion). Best for checking connectivity or finding cycles.

4.  **Dynamic Programming ($\text{DP}$):**
    *   **The Concept:** This is an optimization technique used when a problem has **overlapping subproblems** and **optimal substructure**.
    *   **How it works:** Instead of calculating the solution to the same smaller piece of the problem multiple times (leading to exponential time, $\text{O}(2^n)$), DP uses **memoization** (storing the results of subproblems in a cache/map) so it only calculates them once.
    *   **When to use it:** When you see sequence-based problems involving maximizing or counting possibilities (e.g., Fibonacci sequence, Longest Common Subsequence).

***

## 🚀 The Engineer’s Roadmap: Prioritization for Advancement

If you only have time to deeply study *three* areas, these are the highest leverage points for advancing a career:

### 🥇 Tier 1: Absolute Mastery (The Daily Toolkit)

1.  **Big $\text{O}$ Notation:** Spend time mastering this. Before writing a single line of code for a complex problem, talk through the runtime in terms of Big $\text{O}$.
2.  **Hash Maps / Hash Tables:** Understand collisions, and the trade-offs of different collision resolution techniques. This is used everywhere.
3.  **Graphs & Traversals ($\text{BFS}$/$\text{DFS}$):** The ability to model real-world relationships (social connections, routes, dependencies) using a graph is what moves you beyond simple CRUD applications into complex systems design.

### 🥈 Tier 2: Advanced Proficiency (System Design Depth)

1.  **Trees (Especially Self-Balancing):** Understand *why* a standard BST degrades and what RB-Trees solve. This shows deep knowledge of data integrity.
2.  **Dynamic Programming:** Being able to recognize that a complex counting or optimization problem *can* be solved with DP is a hallmark of an advanced engineer.

### 🥉 Tier 3: Specialized Tools (Deep Dives)

1.  **Tries (Prefix Trees):** Essential for autocomplete, spell checkers, and IP routing.
2.  **Heaps (Priority Queues):** Excellent for "Top K" problems (e.g., finding the 3 most popular items right now).

***

In summary, my advice is this: Do not treat these topics as isolated chapters in a textbook. Treat them as **tools in a mechanical toolbox.**

When you encounter a problem, do not ask, "What data structure should I use?" Instead, ask this sequence of questions:

1.  **What is the core object?** (Is it a sequence? A map? A network?) $\rightarrow$ *This suggests the general structure (List, Map, Graph).*
2.  **What operations must be efficient?** (Need fast searching? Need easy additions?) $\rightarrow$ *This points to the best specific structure (Hash Map vs. Sorted Array).*
3.  **What pattern am I following?** (Am I exploring neighbors? Am I solving overlapping subproblems?) $\rightarrow$ *This dictates the algorithm (BFS vs. DP).*

Mastering this **thought process** is what will truly advance your career.

Any questions?